In [1]:
# Importa tudo

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

In [40]:
def lista_tipos_pedidos ():
    with closing(conn.cursor()) as cursor:
        cursor.execute("SELECT id, resumo FROM pedidos")
        pedidos_lista = [f"ID: {row[0]}. RESUMO : {row[1]}" for row in cursor.fetchall() if row[1]]

    tipos_pedidos = "\n".join(pedidos_lista)
    return tipos_pedidos

In [41]:
tipos_pedidos = lista_tipos_pedidos()
tipos_pedidos

'ID: 1. RESUMO : Pedido de desistência do processo\nID: 2. RESUMO : Informação de que a parte está ciente.'

In [30]:
pedido = "Informação de que a parte está ciente."
with closing(conn.cursor()) as cursor:
    cursor.execute(
        "INSERT INTO pedidos (resumo) VALUES (?)",
        (pedido,)
    )
    conn.commit()

In [36]:
# Seleciona os primeiros 10 documentos que contenham "*PET*" no texto
documentos_pet_filtrados = [doc for doc in documentos_pet if "*PET*" in doc][:10]
documentos_pet_filtrados

['**********PET**********\n\r\nPoder Judiciário\r\nTribunal de Justiça do Estado do Rio Grande do Sul\r\nVara Judicial da Comarca de Sarandi\r\nRua Senador Alberto Pasqualini, 1211 - Bairro: Centro - CEP: 99560000 - Fone: (54) 3046-9896 - Email: frsarandivjud@tjrs.jus.br\r\n\r\n\r\nEXECUÇÃO FISCAL Nº 5000005-84.2008.8.21.0069/RS\r\n\r\nEXEQUENTE: MUNICÍPIO DE SARANDI / RS\r\n\r\nEXECUTADO: NELSI TERESINHA OLIARI\r\n\r\nEXECUTADO: LUCIANA OLIARI DE OLIVEIRA PETRY\r\n\r\nEXECUTADO: ALGARINO ERD DE OLIVEIRA\r\n\r\nLocal: Sarandi\r\n\r\nData: 26/04/2023\r\n\r\nOFÍCIO Nº 10037205756\r\n\r\n(Ao responder, favor mencionar o nº do processo e remeter para o e-mail setorial frsarandivjud@tjrs.jus.br)\r\n\r\nSenhor(a),\r\n\r\nSolicito a Vossa Senhoria providências no sentido de incluir em seus cadastros de inadimplentes a executada LUCIANA OLIARI DE OLIVEIRA PETRY, inscrita no CPF nº 00421393092, residente e domiciliada na RUA PADRE AUGUSTO BATTAION, 585 - VICENTINOS - 99560000, Sarandi/RS, e NEL

In [71]:
# Faz um resumo do conteúdo usando o Gemma + as minutas da Vara
peticao = documentos_pet_filtrados[0]
print(peticao)


**********PET**********

Poder Judiciário
Tribunal de Justiça do Estado do Rio Grande do Sul
Vara Judicial da Comarca de Sarandi
Rua Senador Alberto Pasqualini, 1211 - Bairro: Centro - CEP: 99560000 - Fone: (54) 3046-9896 - Email: frsarandivjud@tjrs.jus.br


EXECUÇÃO FISCAL Nº 5000005-84.2008.8.21.0069/RS

EXEQUENTE: MUNICÍPIO DE SARANDI / RS

EXECUTADO: NELSI TERESINHA OLIARI

EXECUTADO: LUCIANA OLIARI DE OLIVEIRA PETRY

EXECUTADO: ALGARINO ERD DE OLIVEIRA

Local: Sarandi

Data: 26/04/2023

OFÍCIO Nº 10037205756

(Ao responder, favor mencionar o nº do processo e remeter para o e-mail setorial frsarandivjud@tjrs.jus.br)

Senhor(a),

Solicito a Vossa Senhoria providências no sentido de incluir em seus cadastros de inadimplentes a executada LUCIANA OLIARI DE OLIVEIRA PETRY, inscrita no CPF nº 00421393092, residente e domiciliada na RUA PADRE AUGUSTO BATTAION, 585 - VICENTINOS - 99560000, Sarandi/RS, e NELSI TERESINHA OLIARI, CPF 65116623068, residente e domiciliada na Rua Padre Augusto B

In [76]:
def ollama_resumo(pedido):
    pergunta_gemma = "Considere o seguinte pedido, identificado por *PET* (os outros pedaços do texto são apenas anexos)" \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. O resumo deve ser genérico e breve (uma frase apenas)." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])


In [77]:

for pedido in documentos_pet_filtrados:
    print(ollama_resumo(pedido))

Solicita-se a inclusão de executadas em cadastros de inadimplentes para fins de cobrança de dívida tributária.

Requer-se a manutenção da sentença que extingue o processo executivo por ausência de interesse de agir, considerando a ineficiência e a falta de perspectiva de solução, e a condenação do recorrente em honorários de sucumbência.
A parte Executada solicita a baixa das restrições impostas em seu nome no Sistema Renajud e em outros cadastros relacionados à execução fiscal.


In [70]:
print(resumo)

Solicita-se a inclusão de executadas em cadastros de inadimplentes para fins de cobrança de dívida pública.

